# Advanced Solution — Lazy CSV Data Pipeline

This notebook implements all four project goals using:

- lazy, reusable iterator factories;
- `csv.DictReader` rather than fragile string splitting;
- explicit `NamedTuple` record types;
- robust type conversion and contextual error messages;
- equal-length and optional SSN-alignment validation;
- a non-stale-record iterator;
- tie-aware aggregation of vehicle makes by gender;
- lightweight, self-contained tests.

The four CSV files are expected to be in the same directory as this notebook.


## 1. Imports and typed record definitions

Identifiers such as SSNs and employee IDs remain strings. Model years are integers,
and timestamps are normalized to timezone-aware UTC `datetime` objects.


In [1]:
from __future__ import annotations

import csv
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import islice, zip_longest
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import (
    Any,
    Callable,
    Iterable,
    Iterator,
    Mapping,
    NamedTuple,
    Type,
    TypeVar,
    Union,
)


class PersonalInfo(NamedTuple):
    ssn: str
    first_name: str
    last_name: str
    gender: str
    language: str


class Vehicle(NamedTuple):
    ssn: str
    vehicle_make: str
    vehicle_model: str
    model_year: int


class Employment(NamedTuple):
    ssn: str
    employer: str
    department: str
    employee_id: str


class UpdateStatus(NamedTuple):
    ssn: str
    last_updated: datetime
    created: datetime


class PersonRecord(NamedTuple):
    ssn: str
    first_name: str
    last_name: str
    gender: str
    language: str
    vehicle_make: str
    vehicle_model: str
    model_year: int
    employer: str
    department: str
    employee_id: str
    last_updated: datetime
    created: datetime


class MakeGroup(NamedTuple):
    gender: str
    vehicle_makes: tuple[str, ...]
    count: int


RecordT = TypeVar("RecordT", bound=tuple)
Converter = Callable[[str], Any]


## 2. Cleaning and conversion helpers

The timestamp parser first handles normal ISO-8601 values, including a trailing
`Z`, and then tries several common fallback formats. Every returned timestamp is
timezone-aware and normalized to UTC, which makes comparisons safe and predictable.


In [2]:
def clean_text(value: str) -> str:
    """Trim surrounding whitespace while preserving the original text."""
    if value is None:
        raise ValueError("expected text, received None")
    return value.strip()


def parse_int(value: str) -> int:
    """Parse an integer after removing surrounding whitespace."""
    return int(clean_text(value))


def parse_datetime(value: str) -> datetime:
    """Parse a timestamp and normalize it to an aware UTC datetime."""
    raw = clean_text(value)
    if not raw:
        raise ValueError("timestamp cannot be empty")

    # Python's fromisoformat understands offsets, but a trailing Z is normalized
    # explicitly for compatibility with Python 3.9.
    iso_candidate = raw[:-1] + "+00:00" if raw.endswith(("Z", "z")) else raw

    try:
        parsed = datetime.fromisoformat(iso_candidate)
    except ValueError:
        fallback_formats = (
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d %H:%M:%S%z",
            "%Y-%m-%dT%H:%M:%S",
            "%Y-%m-%dT%H:%M:%S%z",
            "%m/%d/%Y %H:%M:%S",
            "%m/%d/%Y",
            "%Y-%m-%d",
        )

        for fmt in fallback_formats:
            try:
                parsed = datetime.strptime(raw, fmt)
                break
            except ValueError:
                continue
        else:
            raise ValueError(f"unsupported timestamp format: {raw!r}")

    if parsed.tzinfo is None:
        return parsed.replace(tzinfo=timezone.utc)

    return parsed.astimezone(timezone.utc)


## 3. Generic lazy CSV reader

`iter_typed_csv` opens a file only when iteration starts, validates its header,
converts each field, and yields one typed named tuple at a time. The whole file is
never loaded into memory.

Using `utf-8-sig` also handles an optional UTF-8 byte-order mark in the first header.


In [3]:
def iter_typed_csv(
    path: Union[str, Path],
    record_type: Type[RecordT],
    converters: Mapping[str, Converter],
) -> Iterator[RecordT]:
    """Yield validated, converted rows from *path* as *record_type* instances."""
    path = Path(path)
    expected_fields = tuple(record_type._fields)

    missing_converters = set(expected_fields) - set(converters)
    unknown_converters = set(converters) - set(expected_fields)
    if missing_converters or unknown_converters:
        raise ValueError(
            "converter schema mismatch: "
            f"missing={sorted(missing_converters)}, "
            f"unknown={sorted(unknown_converters)}"
        )

    with path.open("r", encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source)

        if reader.fieldnames is None:
            raise ValueError(f"{path}: CSV header is missing")

        normalized_headers = [header.strip() for header in reader.fieldnames]
        if len(normalized_headers) != len(set(normalized_headers)):
            raise ValueError(f"{path}: duplicate column names in CSV header")

        # DictReader uses this list when producing subsequent row dictionaries.
        reader.fieldnames = normalized_headers

        actual_fields = set(normalized_headers)
        expected_field_set = set(expected_fields)
        missing_fields = expected_field_set - actual_fields
        extra_fields = actual_fields - expected_field_set

        if missing_fields or extra_fields:
            raise ValueError(
                f"{path}: unexpected CSV schema; "
                f"missing={sorted(missing_fields)}, "
                f"extra={sorted(extra_fields)}"
            )

        for raw_row in reader:
            # Ignore completely blank physical lines.
            if not any(str(value).strip() for value in raw_row.values() if value is not None):
                continue

            converted: dict[str, Any] = {}
            try:
                for field in expected_fields:
                    raw_value = raw_row[field]
                    if raw_value is None:
                        raise ValueError(f"missing value for field {field!r}")
                    converted[field] = converters[field](raw_value)
            except (TypeError, ValueError) as exc:
                raise ValueError(
                    f"{path}, CSV line {reader.line_num}: {exc}"
                ) from exc

            yield record_type(**converted)


## 4. File schemas and iterator factories — Goal 1

These functions are factories rather than stored, one-shot generator objects.
Calling a factory creates a fresh independent iterator, so the same pipeline can be
traversed again without manually reopening files.


In [4]:
PERSONAL_CONVERTERS: Mapping[str, Converter] = {
    "ssn": clean_text,
    "first_name": clean_text,
    "last_name": clean_text,
    "gender": clean_text,
    "language": clean_text,
}

VEHICLE_CONVERTERS: Mapping[str, Converter] = {
    "ssn": clean_text,
    "vehicle_make": clean_text,
    "vehicle_model": clean_text,
    "model_year": parse_int,
}

EMPLOYMENT_CONVERTERS: Mapping[str, Converter] = {
    "ssn": clean_text,
    "employer": clean_text,
    "department": clean_text,
    "employee_id": clean_text,
}

UPDATE_STATUS_CONVERTERS: Mapping[str, Converter] = {
    "ssn": clean_text,
    "last_updated": parse_datetime,
    "created": parse_datetime,
}


def iter_personal_info(data_dir: Union[str, Path] = ".") -> Iterator[PersonalInfo]:
    return iter_typed_csv(
        Path(data_dir) / "personal_info.csv",
        PersonalInfo,
        PERSONAL_CONVERTERS,
    )


def iter_vehicles(data_dir: Union[str, Path] = ".") -> Iterator[Vehicle]:
    return iter_typed_csv(
        Path(data_dir) / "vehicles.csv",
        Vehicle,
        VEHICLE_CONVERTERS,
    )


def iter_employment(data_dir: Union[str, Path] = ".") -> Iterator[Employment]:
    return iter_typed_csv(
        Path(data_dir) / "employment.csv",
        Employment,
        EMPLOYMENT_CONVERTERS,
    )


def iter_update_status(data_dir: Union[str, Path] = ".") -> Iterator[UpdateStatus]:
    return iter_typed_csv(
        Path(data_dir) / "update_status.csv",
        UpdateStatus,
        UPDATE_STATUS_CONVERTERS,
    )


In [5]:
DATA_DIR = Path(".")

required_files = {
    "personal_info.csv",
    "vehicles.csv",
    "employment.csv",
    "update_status.csv",
}
missing_files = sorted(
    file_name for file_name in required_files
    if not (DATA_DIR / file_name).is_file()
)

if missing_files:
    print(
        "Place the project CSV files next to this notebook before running "
        f"the data cells. Missing: {missing_files}"
    )
else:
    print("Personal information sample:")
    for record in islice(iter_personal_info(DATA_DIR), 3):
        print(record)

    print("\nVehicle sample:")
    for record in islice(iter_vehicles(DATA_DIR), 3):
        print(record)

    print("\nEmployment sample:")
    for record in islice(iter_employment(DATA_DIR), 3):
        print(record)

    print("\nUpdate-status sample:")
    for record in islice(iter_update_status(DATA_DIR), 3):
        print(record)


Place the project CSV files next to this notebook before running the data cells. Missing: ['employment.csv', 'personal_info.csv', 'update_status.csv', 'vehicles.csv']


## 5. Combine all four streams — Goal 2

The project guarantees equal lengths and aligned SSNs. The implementation still
checks both conditions because silent truncation by ordinary `zip` is a common and
difficult-to-detect data-quality bug.

Set `verify_ssn=False` only when the upstream guarantees are fully trusted.


In [6]:
_SENTINEL = object()


def zip_equal(*iterables: Iterable[Any]) -> Iterator[tuple[Any, ...]]:
    """Zip iterables lazily and raise if their lengths differ."""
    for row_number, values in enumerate(
        zip_longest(*iterables, fillvalue=_SENTINEL),
        start=1,
    ):
        if any(value is _SENTINEL for value in values):
            raise ValueError(
                "input files contain different numbers of data rows; "
                f"mismatch detected at logical row {row_number}"
            )
        yield values


def iter_people(
    data_dir: Union[str, Path] = ".",
    *,
    verify_ssn: bool = True,
) -> Iterator[PersonRecord]:
    """Yield one combined record per person without repeating the SSN."""
    streams = (
        iter_personal_info(data_dir),
        iter_vehicles(data_dir),
        iter_employment(data_dir),
        iter_update_status(data_dir),
    )

    for row_number, rows in enumerate(zip_equal(*streams), start=1):
        personal, vehicle, employment, status = rows

        if verify_ssn:
            ssns = {
                personal.ssn,
                vehicle.ssn,
                employment.ssn,
                status.ssn,
            }
            if len(ssns) != 1:
                raise ValueError(
                    f"SSN mismatch at logical row {row_number}: "
                    f"{personal.ssn!r}, {vehicle.ssn!r}, "
                    f"{employment.ssn!r}, {status.ssn!r}"
                )

        yield PersonRecord(
            ssn=personal.ssn,
            first_name=personal.first_name,
            last_name=personal.last_name,
            gender=personal.gender,
            language=personal.language,
            vehicle_make=vehicle.vehicle_make,
            vehicle_model=vehicle.vehicle_model,
            model_year=vehicle.model_year,
            employer=employment.employer,
            department=employment.department,
            employee_id=employment.employee_id,
            last_updated=status.last_updated,
            created=status.created,
        )


In [7]:
if not missing_files:
    print("Combined-record sample:")
    for record in islice(iter_people(DATA_DIR), 5):
        print(record)


## 6. Current records only — Goal 3

A record is stale when `last_updated < 2017-03-01`. Therefore a timestamp exactly
at the cutoff is current and is retained.


In [8]:
STALE_CUTOFF = datetime(2017, 3, 1, tzinfo=timezone.utc)


def iter_current_people(
    data_dir: Union[str, Path] = ".",
    *,
    cutoff: datetime = STALE_CUTOFF,
    verify_ssn: bool = True,
) -> Iterator[PersonRecord]:
    """Yield combined records whose last update is on or after *cutoff*."""
    if cutoff.tzinfo is None:
        cutoff = cutoff.replace(tzinfo=timezone.utc)
    else:
        cutoff = cutoff.astimezone(timezone.utc)

    return (
        record
        for record in iter_people(data_dir, verify_ssn=verify_ssn)
        if record.last_updated >= cutoff
    )


In [9]:
if not missing_files:
    total_count = sum(1 for _ in iter_people(DATA_DIR))
    current_count = sum(1 for _ in iter_current_people(DATA_DIR))

    print(f"Total records:   {total_count:,}")
    print(f"Current records: {current_count:,}")
    print(f"Stale records:   {total_count - current_count:,}")


## 7. Largest vehicle-make group for each gender — Goal 4

The aggregation consumes the current-record iterator once and keeps only counters,
not full person records. Every make tied for the maximum count is retained.
Results are sorted for deterministic output.


In [10]:
def largest_make_groups(
    records: Iterable[PersonRecord],
) -> tuple[MakeGroup, ...]:
    """Return all maximum-frequency vehicle makes for every gender."""
    counts_by_gender: dict[str, Counter[str]] = defaultdict(Counter)

    for record in records:
        counts_by_gender[record.gender][record.vehicle_make] += 1

    results: list[MakeGroup] = []

    for gender in sorted(counts_by_gender):
        make_counts = counts_by_gender[gender]
        max_count = max(make_counts.values())
        winners = tuple(
            sorted(
                make
                for make, count in make_counts.items()
                if count == max_count
            )
        )
        results.append(
            MakeGroup(
                gender=gender,
                vehicle_makes=winners,
                count=max_count,
            )
        )

    return tuple(results)


In [11]:
if not missing_files:
    largest_groups = largest_make_groups(
        iter_current_people(DATA_DIR)
    )

    for group in largest_groups:
        makes = ", ".join(group.vehicle_makes)
        print(
            f"{group.gender}: {makes} "
            f"({group.count:,} current record(s) each)"
        )


## 8. Dataset audit and complementary stale iterator

The project data is guaranteed to be aligned, but production pipelines should make
those assumptions observable. The optional audit checks record freshness, duplicate
or out-of-order SSNs, SSN formatting, blank string fields, impossible timestamp
chronology, and future update timestamps.

The audit is separate from the core lazy pipeline. It consumes one fresh iterator and
uses a set of SSNs to validate uniqueness, trading `O(n)` memory for stronger integrity
validation.


In [12]:
import hashlib
import hmac
import json
import re
from tempfile import NamedTemporaryFile


SSN_PATTERN = re.compile(r"^\d{3}-\d{2}-\d{4}$")


class DatasetAudit(NamedTuple):
    total_records: int
    current_records: int
    stale_records: int
    duplicate_ssns: int
    out_of_order_ssns: int
    malformed_ssns: int
    blank_string_values: int
    created_after_updated: int
    future_last_updated: int
    earliest_created: datetime | None
    latest_last_updated: datetime | None

    @property
    def integrity_issues(self) -> int:
        return (
            self.duplicate_ssns
            + self.out_of_order_ssns
            + self.malformed_ssns
            + self.blank_string_values
            + self.created_after_updated
            + self.future_last_updated
        )

    @property
    def is_clean(self) -> bool:
        return self.integrity_issues == 0


def as_utc(value: datetime) -> datetime:
    """Return *value* as a timezone-aware UTC datetime."""
    if value.tzinfo is None:
        return value.replace(tzinfo=timezone.utc)
    return value.astimezone(timezone.utc)


def iter_stale_people(
    data_dir: str | Path = ".",
    *,
    cutoff: datetime = STALE_CUTOFF,
    verify_ssn: bool = True,
) -> Iterator[PersonRecord]:
    """Yield combined records whose last update is strictly before *cutoff*."""
    normalized_cutoff = as_utc(cutoff)
    return (
        record
        for record in iter_people(data_dir, verify_ssn=verify_ssn)
        if record.last_updated < normalized_cutoff
    )


def audit_records(
    records: Iterable[PersonRecord],
    *,
    cutoff: datetime = STALE_CUTOFF,
    audit_time: datetime | None = None,
) -> DatasetAudit:
    """Consume *records* once and return an integrity and freshness report."""
    normalized_cutoff = as_utc(cutoff)
    normalized_audit_time = as_utc(
        audit_time if audit_time is not None else datetime.now(timezone.utc)
    )

    total_records = 0
    current_records = 0
    stale_records = 0
    duplicate_ssns = 0
    out_of_order_ssns = 0
    malformed_ssns = 0
    blank_string_values = 0
    created_after_updated = 0
    future_last_updated = 0
    earliest_created: datetime | None = None
    latest_last_updated: datetime | None = None

    seen_ssns: set[str] = set()
    previous_ssn: str | None = None

    for record in records:
        total_records += 1

        if record.last_updated >= normalized_cutoff:
            current_records += 1
        else:
            stale_records += 1

        if record.ssn in seen_ssns:
            duplicate_ssns += 1
        else:
            seen_ssns.add(record.ssn)

        if previous_ssn is not None and record.ssn < previous_ssn:
            out_of_order_ssns += 1
        previous_ssn = record.ssn

        if SSN_PATTERN.fullmatch(record.ssn) is None:
            malformed_ssns += 1

        blank_string_values += sum(
            1
            for value in record
            if isinstance(value, str) and not value.strip()
        )

        if record.created > record.last_updated:
            created_after_updated += 1

        if record.last_updated > normalized_audit_time:
            future_last_updated += 1

        earliest_created = (
            record.created
            if earliest_created is None
            else min(earliest_created, record.created)
        )
        latest_last_updated = (
            record.last_updated
            if latest_last_updated is None
            else max(latest_last_updated, record.last_updated)
        )

    return DatasetAudit(
        total_records=total_records,
        current_records=current_records,
        stale_records=stale_records,
        duplicate_ssns=duplicate_ssns,
        out_of_order_ssns=out_of_order_ssns,
        malformed_ssns=malformed_ssns,
        blank_string_values=blank_string_values,
        created_after_updated=created_after_updated,
        future_last_updated=future_last_updated,
        earliest_created=earliest_created,
        latest_last_updated=latest_last_updated,
    )


In [13]:
if not missing_files:
    audit = audit_records(iter_people(DATA_DIR))

    print("Dataset audit")
    print("-------------")
    for field, value in audit._asdict().items():
        print(f"{field:>24}: {value}")
    print(f"{'is_clean':>24}: {audit.is_clean}")


## 9. One-pass summaries and privacy-conscious exports

`summarize_records` retains only counters and running statistics. Its memory use is
therefore independent of the number of people, apart from distinct category values.

Because SSNs are sensitive, the CSV exporter accepts a caller-supplied HMAC transform.
This creates stable pseudonyms for joins without writing raw SSNs. It is
**pseudonymization, not anonymization**: the secret and exports still require protection.
Exports are written atomically through a temporary file in the destination directory.


In [14]:
def summarize_records(
    records: Iterable[PersonRecord],
    *,
    top_n: int = 10,
) -> dict[str, Any]:
    """Build a compact JSON-serializable summary in one pass."""
    if top_n < 1:
        raise ValueError("top_n must be at least 1")

    record_count = 0
    gender_counts: Counter[str] = Counter()
    make_counts: Counter[str] = Counter()
    employer_counts: Counter[str] = Counter()
    min_model_year: int | None = None
    max_model_year: int | None = None
    model_year_total = 0
    earliest_update: datetime | None = None
    latest_update: datetime | None = None

    for record in records:
        record_count += 1
        gender_counts[record.gender] += 1
        make_counts[record.vehicle_make] += 1
        employer_counts[record.employer] += 1
        model_year_total += record.model_year

        min_model_year = (
            record.model_year
            if min_model_year is None
            else min(min_model_year, record.model_year)
        )
        max_model_year = (
            record.model_year
            if max_model_year is None
            else max(max_model_year, record.model_year)
        )
        earliest_update = (
            record.last_updated
            if earliest_update is None
            else min(earliest_update, record.last_updated)
        )
        latest_update = (
            record.last_updated
            if latest_update is None
            else max(latest_update, record.last_updated)
        )

    average_model_year = (
        round(model_year_total / record_count, 2)
        if record_count
        else None
    )

    def top_items(counter: Counter[str]) -> list[dict[str, Any]]:
        return [
            {"value": value, "count": count}
            for value, count in sorted(
                counter.items(),
                key=lambda item: (-item[1], item[0]),
            )[:top_n]
        ]

    return {
        "record_count": record_count,
        "gender_counts": dict(sorted(gender_counts.items())),
        "top_vehicle_makes": top_items(make_counts),
        "top_employers": top_items(employer_counts),
        "model_year": {
            "minimum": min_model_year,
            "maximum": max_model_year,
            "average": average_model_year,
        },
        "last_updated": {
            "earliest": (
                earliest_update.isoformat()
                if earliest_update is not None
                else None
            ),
            "latest": (
                latest_update.isoformat()
                if latest_update is not None
                else None
            ),
        },
    }


def make_ssn_pseudonymizer(
    secret: bytes,
    *,
    output_length: int = 24,
) -> Callable[[str], str]:
    """Create a stable HMAC-SHA256 SSN pseudonymizer."""
    if not secret:
        raise ValueError("secret must not be empty")
    if not 12 <= output_length <= 64:
        raise ValueError("output_length must be between 12 and 64")

    def pseudonymize(ssn: str) -> str:
        digest = hmac.new(
            secret,
            ssn.encode("utf-8"),
            hashlib.sha256,
        ).hexdigest()
        return digest[:output_length]

    return pseudonymize


def _temporary_output_path(destination: Path) -> tuple[Path, Any]:
    """Return a same-directory temporary path and its open text handle."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    handle = NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        newline="",
        dir=destination.parent,
        prefix=f".{destination.name}.",
        suffix=".tmp",
        delete=False,
    )
    return Path(handle.name), handle


def export_records_csv(
    records: Iterable[PersonRecord],
    destination: str | Path,
    *,
    ssn_transform: Callable[[str], str] | None = None,
) -> Path:
    """Atomically export records to CSV, optionally pseudonymizing SSNs."""
    destination = Path(destination)
    temp_path, handle = _temporary_output_path(destination)

    try:
        with handle:
            writer = csv.DictWriter(
                handle,
                fieldnames=list(PersonRecord._fields),
            )
            writer.writeheader()

            for record in records:
                row = record._asdict()
                if ssn_transform is not None:
                    row["ssn"] = ssn_transform(record.ssn)
                row["last_updated"] = record.last_updated.isoformat()
                row["created"] = record.created.isoformat()
                writer.writerow(row)

        temp_path.replace(destination)
    except Exception:
        temp_path.unlink(missing_ok=True)
        raise

    return destination


def export_summary_json(
    summary: Mapping[str, Any],
    destination: str | Path,
) -> Path:
    """Atomically write a summary mapping as formatted UTF-8 JSON."""
    destination = Path(destination)
    temp_path, handle = _temporary_output_path(destination)

    try:
        with handle:
            json.dump(
                summary,
                handle,
                ensure_ascii=False,
                indent=2,
                sort_keys=True,
            )
            handle.write("\n")

        temp_path.replace(destination)
    except Exception:
        temp_path.unlink(missing_ok=True)
        raise

    return destination


In [15]:
if not missing_files:
    current_summary = summarize_records(
        iter_current_people(DATA_DIR),
        top_n=5,
    )
    print(json.dumps(current_summary, indent=2, ensure_ascii=False))


Example opt-in exports:

```python
export_dir = Path("exports")

# Summary contains no row-level SSNs.
export_summary_json(
    summarize_records(iter_current_people(DATA_DIR)),
    export_dir / "current_people_summary.json",
)

# Generate this secret outside the notebook and store it securely.
pseudonymize = make_ssn_pseudonymizer(b"replace-with-a-secure-secret")
export_records_csv(
    iter_current_people(DATA_DIR),
    export_dir / "current_people_pseudonymized.csv",
    ssn_transform=pseudonymize,
)
```


## 10. Self-contained tests

These tests create temporary CSV files, so they do not depend on the project data.
They verify type conversion, header-order independence, merging, cutoff-boundary
behavior, and ties in the final aggregation.


In [16]:
def write_csv(
    path: Path,
    fieldnames: list[str],
    rows: list[dict[str, Any]],
) -> None:
    with path.open("w", encoding="utf-8", newline="") as destination:
        writer = csv.DictWriter(destination, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


with TemporaryDirectory() as temp_dir_name:
    temp_dir = Path(temp_dir_name)

    write_csv(
        temp_dir / "personal_info.csv",
        ["language", "gender", "last_name", "first_name", "ssn"],
        [
            {
                "ssn": "111-11-1111",
                "first_name": "Ada",
                "last_name": "Lovelace",
                "gender": "Female",
                "language": "English",
            },
            {
                "ssn": "222-22-2222",
                "first_name": "Grace",
                "last_name": "Hopper",
                "gender": "Female",
                "language": "English",
            },
            {
                "ssn": "333-33-3333",
                "first_name": "Alan",
                "last_name": "Turing",
                "gender": "Male",
                "language": "English",
            },
            {
                "ssn": "444-44-4444",
                "first_name": "Edsger",
                "last_name": "Dijkstra",
                "gender": "Male",
                "language": "Dutch",
            },
        ],
    )

    write_csv(
        temp_dir / "vehicles.csv",
        ["ssn", "vehicle_make", "vehicle_model", "model_year"],
        [
            {
                "ssn": "111-11-1111",
                "vehicle_make": "Ford",
                "vehicle_model": "A",
                "model_year": "2018",
            },
            {
                "ssn": "222-22-2222",
                "vehicle_make": "Tesla",
                "vehicle_model": "B",
                "model_year": "2020",
            },
            {
                "ssn": "333-33-3333",
                "vehicle_make": "Ford",
                "vehicle_model": "C",
                "model_year": "2017",
            },
            {
                "ssn": "444-44-4444",
                "vehicle_make": "Tesla",
                "vehicle_model": "D",
                "model_year": "2019",
            },
        ],
    )

    write_csv(
        temp_dir / "employment.csv",
        ["department", "employee_id", "employer", "ssn"],
        [
            {
                "ssn": "111-11-1111",
                "employer": "One",
                "department": "R&D",
                "employee_id": "001",
            },
            {
                "ssn": "222-22-2222",
                "employer": "Two",
                "department": "Engineering",
                "employee_id": "002",
            },
            {
                "ssn": "333-33-3333",
                "employer": "Three",
                "department": "Research",
                "employee_id": "003",
            },
            {
                "ssn": "444-44-4444",
                "employer": "Four",
                "department": "Algorithms",
                "employee_id": "004",
            },
        ],
    )

    write_csv(
        temp_dir / "update_status.csv",
        ["created", "last_updated", "ssn"],
        [
            {
                "ssn": "111-11-1111",
                "last_updated": "2017-02-28T23:59:59Z",
                "created": "2016-01-01T00:00:00Z",
            },
            {
                "ssn": "222-22-2222",
                "last_updated": "2017-03-01T00:00:00Z",
                "created": "2016-01-01T00:00:00Z",
            },
            {
                "ssn": "333-33-3333",
                "last_updated": "2018-01-01 12:30:00",
                "created": "2016-01-01 00:00:00",
            },
            {
                "ssn": "444-44-4444",
                "last_updated": "2019-01-01T00:00:00+00:00",
                "created": "2016-01-01T00:00:00+00:00",
            },
        ],
    )

    personal = next(iter_personal_info(temp_dir))
    vehicle = next(iter_vehicles(temp_dir))
    combined = tuple(iter_people(temp_dir))
    current = tuple(iter_current_people(temp_dir))
    groups = largest_make_groups(current)

    assert isinstance(personal, PersonalInfo)
    assert isinstance(vehicle.model_year, int)
    assert vehicle.model_year == 2018
    assert isinstance(combined[0], PersonRecord)
    assert combined[0]._fields.count("ssn") == 1
    assert combined[0].employee_id == "001"  # Identifier keeps leading zeros.
    assert all(record.last_updated.tzinfo is not None for record in combined)

    # The first row is stale; the exact cutoff in the second row is current.
    assert [record.ssn for record in current] == [
        "222-22-2222",
        "333-33-3333",
        "444-44-4444",
    ]

    # Female has one current Tesla. Male has a Ford/Tesla tie.
    assert groups == (
        MakeGroup("Female", ("Tesla",), 1),
        MakeGroup("Male", ("Ford", "Tesla"), 1),
    )

    audit = audit_records(
        iter_people(temp_dir),
        audit_time=datetime(2025, 1, 1, tzinfo=timezone.utc),
    )
    assert audit.total_records == 4
    assert audit.current_records == 3
    assert audit.stale_records == 1
    assert audit.is_clean

    summary = summarize_records(current, top_n=2)
    assert summary["record_count"] == 3
    assert summary["gender_counts"] == {"Female": 1, "Male": 2}
    assert summary["model_year"] == {
        "minimum": 2017,
        "maximum": 2020,
        "average": 2018.67,
    }

    pseudonymize = make_ssn_pseudonymizer(b"unit-test-secret")
    export_path = export_records_csv(
        iter(current),
        temp_dir / "current_export.csv",
        ssn_transform=pseudonymize,
    )
    with export_path.open(encoding="utf-8", newline="") as exported_file:
        exported_rows = list(csv.DictReader(exported_file))

    assert len(exported_rows) == 3
    assert exported_rows[0]["ssn"] != current[0].ssn
    assert exported_rows[0]["ssn"] == pseudonymize(current[0].ssn)
    assert exported_rows[0]["last_updated"].endswith("+00:00")

    summary_path = export_summary_json(
        summary,
        temp_dir / "summary.json",
    )
    loaded_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    assert loaded_summary == summary

    try:
        tuple(zip_equal([1], [2, 3]))
    except ValueError as exc:
        assert "different numbers" in str(exc)
    else:
        raise AssertionError("zip_equal should reject unequal lengths")

    try:
        parse_datetime("not-a-timestamp")
    except ValueError as exc:
        assert "unsupported timestamp format" in str(exc)
    else:
        raise AssertionError("parse_datetime should reject invalid input")

print("All self-contained tests passed.")


All self-contained tests passed.


## 11. Final usage

The primary project result is:

```python
largest_make_groups(iter_current_people(DATA_DIR))
```

Useful operational entry points are:

```python
audit_records(iter_people(DATA_DIR))
summarize_records(iter_current_people(DATA_DIR))
iter_stale_people(DATA_DIR)
```

Every call uses a fresh iterator factory. The CSV pipeline remains lazy; only
counters, audit state, or explicitly requested export data are materialized.
